In [ ]:
import pandas as pd
import numpy as np
import io
import os
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

# Configuration esthétique globale
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("--- ÉTAPE 1 : CHARGEMENT ET CORRECTION DES DONNÉES ---")

# Recherche du fichier spécifique que vous venez de charger
nom_fichier = "US Superstore data (3).xls - Orders.csv"

if os.path.exists(nom_fichier):
    print(f"✓ Fichier trouvé localement : {nom_fichier}")
    # Determine if it's a CSV or Excel based on extension if a file exists locally
    if nom_fichier.endswith('.csv'):
        df = pd.read_csv(nom_fichier, sep=',', encoding='utf-8')
    elif nom_fichier.endswith(('.xls', '.xlsx')):
        df = pd.read_excel(nom_fichier)
    else:
        print("Format de fichier non pris en charge localement.")
        # Fallback to upload if local file is not recognized
        uploaded = files.upload()
        nom_fichier = list(uploaded.keys())[0]
        if nom_fichier.endswith('.csv'):
            df = pd.read_csv(io.BytesIO(uploaded[nom_fichier]), sep=',', encoding='utf-8')
        elif nom_fichier.endswith(('.xls', '.xlsx')):
            df = pd.read_excel(io.BytesIO(uploaded[nom_fichier]))
        else:
            raise ValueError("Uploaded file format not supported. Please upload a .csv or .xls/.xlsx file.")

else:
    print("Fichier non trouvé localement. Veuillez le sélectionner :")
    uploaded = files.upload()
    nom_fichier = list(uploaded.keys())[0]
    # Check file extension to use appropriate pandas reader
    if nom_fichier.endswith('.csv'):
        df = pd.read_csv(io.BytesIO(uploaded[nom_fichier]), sep=',', encoding='utf-8')
    elif nom_fichier.endswith(('.xls', '.xlsx')):
        df = pd.read_excel(io.BytesIO(uploaded[nom_fichier]))
    else:
        raise ValueError("Uploaded file format not supported. Please upload a .csv or .xls/.xlsx file.")

# --- CORRECTION CRITIQUE DES DATES EXCEL ---
# On s'assure que les colonnes de date sont bien au format datetime.
# pd.read_excel gère souvent les dates numériques directement.
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

# Ajout des colonnes pour les analyses temporelles
df['Order Year'] = df['Order Date'].dt.year
df['Order Month-Year'] = df['Order Date'].dt.to_period('M')

print(f"✓ Données chargées avec succès ! Dimensions : {df.shape[0]} lignes, {df.shape[1]} colonnes.")
print(f"✓ Plage des dates corrigée : de {df['Order Date'].min().strftime('%Y-%m-%d')} à {df['Order Date'].max().strftime('%Y-%m-%d')}\n")
print("="*80 + "\n")


# =====================================================================
# INTERACTIF / MATPLOTLIB : TENDANCES DES VENTES AU FIL DES ANS
# =====================================================================
print("--- ÉTAPE 2 : TENDANCES DES VENTES (MATPLOTLIB) ---")
from ipywidgets import interact, Dropdown

monthly_sales = df.groupby(['Order Month-Year', 'Category'])['Sales'].sum().reset_index()
monthly_sales['Date'] = monthly_sales['Order Month-Year'].dt.to_timestamp()

def plot_sales_trends(category='All'):
    plt.figure(figsize=(12, 5))
    if category == 'All':
        total_monthly = df.groupby('Order Month-Year')['Sales'].sum()
        plt.plot(total_monthly.index.to_timestamp(), total_monthly.values, marker='o', color='darkblue', linewidth=2)
        plt.title('Évolution Mensuelle des Ventes Globales', fontsize=14, fontweight='bold')
    else:
        sub_data = monthly_sales[monthly_sales['Category'] == category]
        plt.plot(sub_data['Date'], sub_data['Sales'], marker='o', color='teal', linewidth=2)
        plt.title(f'Évolution Mensuelle des Ventes - Catégorie : {category}', fontsize=14, fontweight='bold')
    plt.xlabel('Date')
    plt.ylabel('Ventes ($)')
    plt.grid(True, alpha=0.3)
    plt.show()

categories = ['All'] + list(df['Category'].unique())
interact(plot_sales_trends, category=Dropdown(options=categories, value='All', description='Catégorie :'))
print("="*80 + "\n")


# =====================================================================
# SEABORN : TOP 10 PRODUITS ET NUAGE DE POINTS (PROFIT VS REMISE)
# =====================================================================
print("--- ÉTAPE 3 : ANALYSE DES PRODUITS ET DES REMISES (SEABORN) ---")

# 1. Top 10 des produits les plus vendus
top_products = df.groupby('Product Name')['Sales'].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(12, 6))
sns.barplot(x=top_products.values, y=top_products.index, palette='rocket')
plt.title('Top 10 des Produits les plus Vendus (Chiffre d\'Affaires)', fontsize=14, fontweight='bold')
plt.xlabel('Ventes Totales ($)')
plt.ylabel('Nom du Produit')
plt.tight_layout()
plt.show()

# 2. Nuage de points Profit vs Discount
plt.figure(figsize=(12, 6))
sns.scatterplot(data=df, x='Discount', y='Profit', hue='Category', alpha=0.6, s=50)
sns.regplot(data=df, x='Discount', y='Profit', scatter=False, color='red', line_kws={'linestyle': '--'})
plt.title('Analyse de l\'Impact des Remises sur le Profit', fontsize=14, fontweight='bold')
plt.xlabel('Taux de Remise (Discount)')
plt.ylabel('Profit ($)')
plt.axhline(y=0, color='black', linestyle='-', alpha=0.5)
plt.tight_layout()
plt.show()
print("="*80 + "\n")


# =====================================================================
# ANALYSE DU MINI-PROJET : PARETO & ETATS (RÉPONSES AUX QUESTIONS)
# =====================================================================
print("--- ÉTAPE 4 : RÉPONSES AUX QUESTIONS DU MINI-PROJET (PARETO) ---")

# 1. Différence New York vs Californie
comp_ny_ca = df[df['State'].isin(['New York', 'California'])].groupby('State')[['Sales', 'Profit']].sum()
print("[COMPARAISON ÉTATS CLÉS]")
print(comp_ny_ca.to_string())
print("\n")

# 2. Client exceptionnel à New York (Plus grand générateur de Profit)
ny_top_client = df[df['State'] == 'New York'].groupby('Customer Name')['Profit'].sum().idxmax()
ny_top_client_profit = df[df['State'] == 'New York'].groupby('Customer Name')['Profit'].sum().max()
print(f"🏆 Client exceptionnel à New York : {ny_top_client} (Profit généré : ${ny_top_client_profit:,.2f})\n")

# 3. Courbe de Pareto des Clients vs Ventes
clients_sales = df.groupby('Customer Name')['Sales'].sum().sort_values(ascending=False).reset_index()
clients_sales['Cumul_Sales'] = clients_sales['Sales'].cumsum()
clients_sales['Pct_Cumul_Sales'] = clients_sales['Cumul_Sales'] / clients_sales['Sales'].sum()
clients_sales['Pct_Clients'] = (clients_sales.index + 1) / len(clients_sales)

plt.figure(figsize=(10, 5))
plt.plot(clients_sales['Pct_Clients'].values, clients_sales['Pct_Cumul_Sales'].values, color='purple', linewidth=2)
plt.axhline(y=0.8, color='r', linestyle='--', label='Seuil 80% des Ventes')
plt.axvline(x=0.2, color='orange', linestyle='--', label='20% des Clients')
plt.title('Courbe de Pareto : Contribution des Clients aux Ventes', fontsize=13, fontweight='bold')
plt.xlabel('Proportion des Clients')
plt.ylabel('Proportion Cumulée des Ventes')
plt.legend()
plt.show()

pct_sales_20 = clients_sales[clients_sales['Pct_Clients'] <= 0.2]['Pct_Cumul_Sales'].iloc[-1] * 100
print(f"📊 Constat Pareto (Ventes) : Les premiers 20% des clients représentent {pct_sales_20:.1f}% des ventes totales.")

### Correlation Analysis: Sales vs. Profit
Let's analyze the correlation between 'Sales' and 'Profit' to understand their relationship.

In [ ]:
print("--- ÉTAPE 5 : ANALYSE DE CORRÉLATION ---")

correlation_matrix = df[['Sales', 'Profit']].corr()
print("Matrice de Corrélation entre Ventes et Profit :\n")
print(correlation_matrix.to_string())

plt.figure(figsize=(6, 5))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Matrice de Corrélation entre Ventes et Profit')
plt.show()

print("="*80 + "\n")